In [8]:
import urllib.request, json

# 试试 preview 版本的 endpoint
base = "https://comtradeapi.un.org/data/v1/get/C/A/HS"

params = {
    "reporterCode": "156",       # 中国
    "period": "2022",
    "partnerCode": "842",        # 美国
    "cmdCode": "AG2",            # 所有2位HS码
    "flowCode": "M,X",
    "subscription-key": "ff3a4b4ef6b74058ab6866d2f6cd79cf"
}

query = "&".join(f"{k}={v}" for k, v in params.items())
url = f"{base}?{query}"
print("请求 URL:", url)

req = urllib.request.Request(url, headers={"Cache-Control": "no-cache"})

with urllib.request.urlopen(req) as resp:
    raw = resp.read().decode("utf-8")
    obj = json.loads(raw)
    print("count:", obj.get("count"))
    print("error:", obj.get("error"))
    # 打印完整返回看看结构
    print(json.dumps(obj, indent=2)[:3000])

请求 URL: https://comtradeapi.un.org/data/v1/get/C/A/HS?reporterCode=156&period=2022&partnerCode=842&cmdCode=AG2&flowCode=M,X&subscription-key=ff3a4b4ef6b74058ab6866d2f6cd79cf
count: 194
error: 
{
  "elapsedTime": "0.52 secs",
  "count": 194,
  "data": [
    {
      "typeCode": "C",
      "freqCode": "A",
      "refPeriodId": 20220101,
      "refYear": 2022,
      "refMonth": 52,
      "period": "2022",
      "reporterCode": 156,
      "reporterISO": null,
      "reporterDesc": null,
      "flowCode": "M",
      "flowDesc": null,
      "partnerCode": 842,
      "partnerISO": null,
      "partnerDesc": null,
      "partner2Code": 0,
      "partner2ISO": null,
      "partner2Desc": null,
      "classificationCode": "H6",
      "classificationSearchCode": "HS",
      "isOriginalClassification": true,
      "cmdCode": "85",
      "cmdDesc": null,
      "aggrLevel": null,
      "isLeaf": null,
      "customsCode": "C00",
      "customsDesc": null,
      "mosCode": "0",
      "motCode": 0,
   

In [10]:
import urllib.request, json, sqlite3, time

API_KEY = "ff3a4b4ef6b74058ab6866d2f6cd79cf"
BASE = "https://comtradeapi.un.org/data/v1/get/C/A/HS"

# 1. 建库建表
conn = sqlite3.connect("comtrade.db")
cur = conn.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS trade (
    reporter_code INT,
    partner_code  INT,
    period        TEXT,
    flow_code     TEXT,
    cmd_code      TEXT,
    primary_value REAL,
    cif_value     REAL,
    fob_value     REAL,
    net_wgt       REAL,
    classification TEXT,
    is_reported   BOOLEAN,
    PRIMARY KEY (reporter_code, partner_code, period, flow_code, cmd_code)
)
""")
conn.commit()

# 2. 从 API 拉数据并写入
def fetch_and_store(reporter, partner, year):
    params = {
        "reporterCode": str(reporter),
        "period": str(year),
        "partnerCode": str(partner),
        "cmdCode": "AG2",
        "flowCode": "M,X",
        "subscription-key": API_KEY
    }
    query = "&".join(f"{k}={v}" for k, v in params.items())
    url = f"{BASE}?{query}"
    req = urllib.request.Request(url, headers={"Cache-Control": "no-cache"})

    with urllib.request.urlopen(req) as resp:
        obj = json.loads(resp.read().decode("utf-8"))

    for r in obj.get("data", []):
        cur.execute("""
            INSERT OR REPLACE INTO trade
            VALUES (?,?,?,?,?,?,?,?,?,?,?)
        """, (
            r["reporterCode"], r["partnerCode"], r["period"],
            r["flowCode"], r["cmdCode"], r["primaryValue"],
            r.get("cifvalue"), r.get("fobvalue"), r.get("netWgt"),
            r.get("classificationCode"), r.get("isReported")
        ))
    conn.commit()
    print(f"  {reporter} -> {partner} ({year}): {obj.get('count',0)} 条")

# 3. 批量拉 SCO 国家数据（示例）
sco = {
    "中国": 156, "俄罗斯": 643, "印度": 356,
    "巴基斯坦": 586, "哈萨克斯坦": 398, "乌兹别克斯坦": 860
}

for name, code in sco.items():
    for partner_name, partner_code in sco.items():
        if code == partner_code:
            continue
        fetch_and_store(code, partner_code, 2022)
        time.sleep(1.5)  # 避免限流

print("数据拉取完成！")

  156 -> 643 (2022): 186 条
  156 -> 356 (2022): 0 条
  156 -> 586 (2022): 175 条
  156 -> 398 (2022): 159 条
  156 -> 860 (2022): 160 条
  643 -> 156 (2022): 0 条
  643 -> 356 (2022): 0 条
  643 -> 586 (2022): 0 条
  643 -> 398 (2022): 0 条
  643 -> 860 (2022): 0 条
  356 -> 156 (2022): 0 条
  356 -> 643 (2022): 0 条
  356 -> 586 (2022): 0 条
  356 -> 398 (2022): 0 条
  356 -> 860 (2022): 0 条
  586 -> 156 (2022): 166 条
  586 -> 643 (2022): 81 条
  586 -> 356 (2022): 0 条
  586 -> 398 (2022): 54 条
  586 -> 860 (2022): 67 条
  398 -> 156 (2022): 462 条
  398 -> 643 (2022): 791 条
  398 -> 356 (2022): 0 条
  398 -> 586 (2022): 124 条
  398 -> 860 (2022): 390 条
  860 -> 156 (2022): 151 条
  860 -> 643 (2022): 181 条
  860 -> 356 (2022): 0 条
  860 -> 586 (2022): 75 条
  860 -> 398 (2022): 168 条
数据拉取完成！


In [11]:
import pandas as pd

# 中国从各SCO国家的总进口额
df = pd.read_sql("""
    SELECT reporter_code, partner_code,
           SUM(primary_value) as total_trade
    FROM trade
    WHERE reporter_code = 156 AND flow_code = 'M'
    GROUP BY reporter_code, partner_code
    ORDER BY total_trade DESC
""", conn)
print(df)

# SCO国家间完整贸易矩阵
df2 = pd.read_sql("""
    SELECT reporter_code, partner_code, flow_code,
           SUM(primary_value) as total_value
    FROM trade
    GROUP BY reporter_code, partner_code, flow_code
""", conn)
print(df2)

   reporter_code  partner_code   total_trade
0            156           643  1.141490e+11
1            156           398  1.481925e+10
2            156           586  3.413278e+09
3            156           860  2.276414e+09
    reporter_code  partner_code flow_code   total_value
0             156           398         M  1.481925e+10
1             156           398         X  1.635524e+10
2             156           586         M  3.413278e+09
3             156           586         X  2.308942e+10
4             156           643         M  1.141490e+11
5             156           643         X  7.612265e+10
6             156           860         M  2.276414e+09
7             156           860         X  7.504177e+09
8             398           156         M  2.953798e+09
9             398           156         X  4.205426e+09
10            398           586         M  2.103062e+07
11            398           586         X  1.154605e+07
12            398           643         M  1.31